# Discretization and assembly

&nbsp;[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/themintlab/ExecutableEngineering/blob/main/chapters/finite_element_method/discretization_and_assembly.ipynb)

### The weak form

Our usual formulation of BVPs is called the *strong* form because it assumes a lot of continuity in a solution. E.g., if the equation contains $\nabla^2 u$, the solution assumes that derivative exists!

The MWR leads to the *weak form* which has fewer requirements on the solution, amongst other benefits. Essentially, all we have to do is multiply through by our test function $v$ and integrate. Something *very* useful happens to our divergence-of-flux terms, which are prevalent in conservation equations.

As an example, consider a time-dependent heat balance (temperature $= u$), using the backward Euler time-marching scheme:

$$\begin{aligned}
\frac{\partial u}{\partial t} &= -\nabla \cdot \mathbf{J} \\
\frac{u - u^{t-1}}{\Delta t} &= -\nabla \cdot \mathbf{J} \\
u + \Delta t (\nabla \cdot \mathbf{J}) &= u^{t-1}
\end{aligned}$$

## Deriving the weak form

To get to the weak form, multiply through by the test function $v$ and integrate:

$$0 = \int_{\Omega} u v \, \mathrm{d}\Omega - \int_{\Omega} u^{t-1} v \, \mathrm{d}\Omega + \Delta t \int_{\Omega} v \nabla \cdot \mathbf{J} \, \mathrm{d}\Omega$$

Examining the divergence-of-flux term, we see that we can apply integration by parts followed by the Divergence Theorem (with $\mathbf{J} = -\lambda \nabla u$):

$$\begin{aligned}
\int_\Omega v \nabla \cdot \mathbf{J} \, \mathrm{d}\Omega &= -\int_\Omega \nabla v \cdot \mathbf{J} \, \mathrm{d}\Omega + \int_{\Omega} \nabla \cdot \big[v \mathbf{J}\big] \, \mathrm{d}\Omega \\
&= -\int_\Omega \nabla v \cdot \mathbf{J} \, \mathrm{d}\Omega + \int_{\partial \Omega} v \, \mathbf{J} \cdot \hat{\mathbf{n}} \, \mathrm{d}S \\
&= \int_\Omega \lambda \nabla v \cdot \nabla u \, \mathrm{d}\Omega + \int_{\partial \Omega} v J_n \, \mathrm{d}S
\end{aligned}$$

where we have defined the outward normal boundary flux, $J_n = \mathbf{J} \cdot \hat{\mathbf{n}}$.

The weak form is remarkable since we:

* replaced a second derivative with the product of two first derivatives.
* no longer require our solution to *have* a second derivative (it can be less smooth and satisfy the equation in an average sense, which is *weaker* than the *strong* form)!
* incorporated boundary fluxes directly into the residual calculation (through the surface integral).
* do not require anything from $\lambda$, which means this is still valid for $\lambda(u, x, t, \dots)$, however it is now outside of the gradient!
* also provided an avenue for directly integrating surface *phenomena* into our model through the surface integral!

## Boundary conditions / terms

The term $\int_{\partial \Omega} v \, \mathbf{J} \cdot \hat{\mathbf{n}} \, \mathrm{d}S$ is a *boundary integral* of the *normal flux*, $J_n = \mathbf{J} \cdot \hat{\mathbf{n}}$.

Typically, we will have some knowledge of the boundary conditions:

* **Neumann (flux)** boundary condition, in which case we have the quantity $J_n$ to insert directly.
* **Insulation (zero flux)**, for which $J_n = 0$ and the term vanishes.
* **Dirichlet (value)** boundary condition, which we insert directly into the applicable unknown, e.g., $u_0 = 0$ for a boundary $u(0) = 0$. In this case, there is still a flux at that boundary, but it is determined by the Dirichlet condition.

Some software will opt to implement Dirichlet conditions as a *weak constraint*, which is a Lagrange multiplier formulation in which the Lagrange multiplier assumes the value of the *reaction force* necessary to enforce that constraint.

## Discretization

We are now in a position to discretize the weak equation,

$$\int_{\Omega} u v \, \mathrm{d}\Omega + \Delta t \int_\Omega \lambda \nabla v \cdot \nabla u \, \mathrm{d}\Omega = \int_{\Omega} u^{t-1} v \, \mathrm{d}\Omega - \Delta t \int_{\partial \Omega} v J_n \, \mathrm{d}S$$

into a form we are able to solve, which will (finally) give us an idea of what the shape functions are.

### Small support for shape functions

The shape functions are generally designed to be non-zero only inside their element. This has the effect of limiting products $\int \phi_i \phi_j \, \mathrm{d}\Omega$ and $\int \nabla \phi_i \cdot \nabla \phi_j \, \mathrm{d}\Omega$ to be zero for almost all combinations of $i$ and $j$, except for cases where the nodes $i$ and $j$ are within the same element!

## Assembly

The matrix assembly stage involves transforming the weak form into a matrix equation for the unknowns $u_i$, such that the approximate solution can be assembled as $u_h = \sum_i u_i \phi_i$.

Consider the term $\int_{\Omega} u v \, \mathrm{d}\Omega$, recalling $u_h = \sum_i u_i \phi_i$ and testing against each basis function $\phi_j$. Since integration is a linear operator, we have:

$$\begin{aligned}
\int_{\Omega} u \phi_j \, \mathrm{d}\Omega &= \sum_i u_i \int_{\Omega} \phi_i \phi_j \, \mathrm{d}\Omega \\
&= \sum_i M_{ji} u_i
\end{aligned}$$

where $\mathbf{M}$ is called the *Mass* matrix.

Similarly,

$$\begin{aligned}
\int_\Omega \lambda \nabla \phi_j \cdot \nabla u \, \mathrm{d}\Omega &= \sum_i u_i \int_{\Omega} \lambda \nabla \phi_j \cdot \nabla \phi_i \, \mathrm{d}\Omega \\
&= \sum_i K_{ji} u_i
\end{aligned}$$

where $\mathbf{K}$ is called the *Stiffness* matrix.

The previous time-step term is

$$\begin{aligned}
\int_{\Omega} u^{t-1} \phi_j \, \mathrm{d}\Omega &= \sum_i u^{t-1}_i \int_{\Omega} \phi_i \phi_j \, \mathrm{d}\Omega \\
&= \sum_i M_{ji} u_i^{t-1} = b_j
\end{aligned}$$

which can also be calculated directly since $u_i^{t-1}$ is known.

## Assembled matrix equation

Finally, we can assemble our equation:

$$\begin{aligned}
\int_{\Omega} u v \, \mathrm{d}\Omega + \Delta t \int_\Omega \lambda \nabla v \cdot \nabla u \, \mathrm{d}\Omega &= \int_{\Omega} u^{t-1} v \, \mathrm{d}\Omega - \Delta t \int_{\partial \Omega} v J_n \, \mathrm{d}S \\
\sum_i M_{ji} u_i + \Delta t \sum_i K_{ji} u_i &= b_j - \Delta t (J_n)_j \\
\mathbf{M} \mathbf{u}_h + \Delta t \mathbf{K} \mathbf{u}_h &= \mathbf{b} - \Delta t \mathbf{J}_n
\end{aligned}$$

The elements of the matrix can be calculated for a reference element and then simply scaled according to the transformation of the template to the mesh. The integration itself can be analytic, or Gaussian Quadrature (for which Gauss Points would be obtained on the reference element).

## Summary of FEM matrices

| **Weak Form Term** | **Matrix Form (FEM)** | **Description** |
|--------------------|-----------------------|-----------------|
| $\int_\Omega \phi_i \phi_j \, \mathrm{d}\Omega$ | $M_{ij}$ | Mass matrix. Often used in time-dependent problems. |
| $\int_\Omega \nabla \phi_i \cdot \nabla \phi_j \, \mathrm{d}\Omega$ | $K_{ij}$ | Stiffness matrix. Used in problems like Poisson or elasticity equations. |
| $\int_\Omega \phi_i f \, \mathrm{d}\Omega$ | $F_i$ | Load vector. Represents source terms in the governing equation. |
| $\int_{\partial \Omega} \phi_i g \, \mathrm{d}S$ | $F_i^{\text{bc}}$ | Boundary load vector (from Neumann boundary conditions). |